# IS THEORY RAG PROTOTYPE

### Import libraries

In [1]:
import os
import json
from IPython.display import JSON

from fastembed import TextEmbedding

import weaviate
from weaviate.classes.data import DataObject

from helper import suppress_output

/Users/jurgen/Documents/deeplearning/deeplearning-rag-prototype/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

### Set variables

In [3]:
COLLECTION_NAME = "Theories"  # capitalize the first letter of collection names
THEORY_FOLDER = "include/data"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

### Instantiate Embedded Weaviate client

In [4]:
with suppress_output():
    client = weaviate.connect_to_embedded(
        persistence_data_path= "tmp/weaviate",
    )
print("Started new embedded Weaviate instance.")
print(f"Client is ready: {client.is_ready()}")

Started new embedded Weaviate instance.
Client is ready: True


### Create the collection of documents

In [5]:
existing_collections = client.collections.list_all()
existing_collection_names = existing_collections.keys()

if COLLECTION_NAME not in existing_collection_names:
    print(f"Collection {COLLECTION_NAME} does not exist yet. Creating it...")
    collection = client.collections.create(name=COLLECTION_NAME)
    print(f"Collection {COLLECTION_NAME} created successfully.")
else:
    print(f"Collection {COLLECTION_NAME} already exists. No action taken.")
    collection = client.collections.get(COLLECTION_NAME)

Collection Theories already exists. No action taken.


### Extract text from local files

In [6]:
# list the theory description files
theory_files = [
    f for f in os.listdir(THEORY_FOLDER)
    if f.endswith('.txt')
]

print(f"Found {len(theory_files)} theory files.")

Found 158 theory files.


You'll now loop through each file. Each file describes **one theory** in Markdown: the theory name is the top-level `#` heading, and `##` sections hold the concise description, the originating author(s), and the seminal articles. You'll parse these sections into one dictionary per theory, treating `N/A` as an empty value.

In [7]:
def parse_theory_file(text):
    """Parse a scraped theory Markdown file into a dict.

    Expected format:
        # <Theory name>
        ## Concise description of theory
        <description or N/A>
        ## Originating author(s)
        <authors or N/A>
        ## Seminal articles
        <articles or N/A>
    """
    name = ""
    sections = {}
    current = None
    buf = []

    def flush():
        if current is not None:
            sections[current] = "\n".join(buf).strip()

    for line in text.splitlines():
        if line.startswith("# ") and not line.startswith("## "):
            name = line[2:].strip()
        elif line.startswith("## "):
            flush()
            current = line[3:].strip().lower()
            buf = []
        elif current is not None:
            buf.append(line)
    flush()

    def clean(value):
        value = (value or "").strip()
        return "" if value == "N/A" else value

    return {
        "name": name,
        "description": clean(sections.get("concise description of theory", "")),
        "authors": clean(sections.get("originating author(s)", "")),
        "seminal_articles": clean(sections.get("seminal articles", "")),
    }


list_of_theory_data = []

for theory_file in theory_files:
    with open(os.path.join(THEORY_FOLDER, theory_file), "r") as f:
        list_of_theory_data.append(parse_theory_file(f.read()))

print(f"Parsed {len(list_of_theory_data)} theories.")

Parsed 158 theories.


In [8]:
JSON(json.dumps(list_of_theory_data))

<IPython.core.display.JSON object>

### Create vector embeddings from theories

In [9]:
embedding_model = TextEmbedding(EMBEDDING_MODEL_NAME)

# Embed the theory name together with its description so that name-only
# theories (with an N/A description) remain searchable by their name.
theory_embeddings = []
for theory in list_of_theory_data:
    text_to_embed = theory["name"]
    if theory["description"]:
        text_to_embed = f'{theory["name"]}. {theory["description"]}'
    theory_embeddings.append(list(embedding_model.embed([text_to_embed]))[0])

### Load embeddings to Weaviate

In [10]:
items = []

for theory, emb in zip(list_of_theory_data, theory_embeddings):
    item = DataObject(
        properties={
            "name": theory["name"],
            "description": theory["description"],
            "authors": theory["authors"],
            "seminal_articles": theory["seminal_articles"],
        },
        vector=emb,
    )
    items.append(item)

collection.data.insert_many(items)

BatchObjectReturn(_all_responses=[UUID('7c4d53aa-92c5-4da6-9299-7686846d8679'), UUID('180b9598-bfb0-4a58-b79f-e1c2f6abc5c4'), UUID('ae1a01d1-6799-4f9c-a5a5-bd410d10becd'), UUID('372c6c74-740d-4353-a986-0f7fefc36723'), UUID('ef289069-55fb-4e83-a516-5f9452e17905'), UUID('9296d7a0-cbad-4f51-9772-d070a2d228cc'), UUID('a22d1dda-69b9-48df-8543-90284957d98f'), UUID('1c565816-5eed-4e67-914e-c872e54a74c4'), UUID('0c8ab14c-0a8f-40d0-b93c-a0a2fd420cf6'), UUID('b165ad7a-172d-4d22-aece-b3433622317f'), UUID('5fad5783-8314-4590-aeff-935500889d2a'), UUID('0163b77c-39f7-42de-a604-8bf7f71912e0'), UUID('2ea36aef-3f8c-4773-9126-36b055159589'), UUID('98bde1fa-db58-4226-adf8-513c838e2008'), UUID('bb0c9ec1-b893-4fb7-bfb3-2abb31d8f92a'), UUID('6cedcb52-8754-4096-a90a-5d5d1dd78a4c'), UUID('ca240b35-2c71-49c6-bc78-ce6f9ca3205b'), UUID('34159b5a-1dae-4128-aecd-530c33338c73'), UUID('3ed6717e-5fc1-41c5-adb1-57937e10512c'), UUID('a6cb4ea2-42c0-4888-85ea-1e7830d57e73'), UUID('a70e1b01-a19d-4f42-ba19-57c135ac5ecd'), 

### Query for a theory recommendation using semantic search

In [11]:
query_str = "A theory explaining why people adopt new technology"

embedding_model = TextEmbedding(EMBEDDING_MODEL_NAME)
collection = client.collections.get(COLLECTION_NAME)

query_emb = list(embedding_model.embed([query_str]))[0]

results = collection.query.near_vector(
    near_vector=query_emb,
    limit=3,
)
for result in results.objects:
    props = result.properties
    print(f"Theory: {props['name']}")
    if props["authors"]:
        print(f"Originating author(s): {props['authors']}")
    if props["description"]:
        print(f"Description: {props['description']}")
    print("-" * 80)

Theory: Technology affordances and constraints theory
Originating author(s): Ann Majchrzak
Lynne Markus
--------------------------------------------------------------------------------
Theory: Technology affordances and constraints theory
Originating author(s): Ann Majchrzak
Lynne Markus
--------------------------------------------------------------------------------
Theory: Unified theory of acceptance and use of technology
Originating author(s): Venkatesh et al. (2003)
Description: The UTAUT aims to explain user intentions to use an IS and subsequent usage behavior. The theory holds that four key constructs (performance expectancy, effort expectancy, social influence, and facilitating conditions) are direct determinants of usage intention and behaviour (Venkatesh et. al., 2003). Gender, age, experience, and voluntariness of use are posited to moderate the impact of the four key constructs on usage intention and behavior (Venkatesh et. al., 2003). The theory was developed through a re

### Optional Cleanup utilities


In [12]:
# ## Remove a collection from an existing Weaviate instance

# client.collections.delete(COLLECTION_NAME)

In [13]:
# ## Delete a Weaviate instance
# ## This cell can take a few seconds to run  

# import shutil

# client.close()

# EMBEDDED_WEAVIATE_PERSISTENCE_PATH = "tmp/weaviate"

# if os.path.exists(EMBEDDED_WEAVIATE_PERSISTENCE_PATH):
#     shutil.rmtree(EMBEDDED_WEAVIATE_PERSISTENCE_PATH)
#     if not os.path.exists(EMBEDDED_WEAVIATE_PERSISTENCE_PATH):
#         print(f"Verified: '{EMBEDDED_WEAVIATE_PERSISTENCE_PATH}' no longer exists.")
#         print(f"Weaviate embedded data at '{EMBEDDED_WEAVIATE_PERSISTENCE_PATH}' deleted.")